<a href="https://colab.research.google.com/github/vermasachin6102/JoyAI-Echo/blob/main/seed_veo_3__joy_ai_echo_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi
import torch
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1024**3
print(f"GPU: {props.name} | VRAM: {vram_gb:.1f} GB")
if vram_gb < 40:
    print("WARNING: <40GB VRAM — JoyAI-Echo will very likely OOM. Switch to G4 or A100 (Runtime > Change runtime type).")
elif vram_gb < 60:
    print("40GB-class GPU: use the reduced settings in Part 2 (num_frames=121, 480x832).")
else:
    print("Big GPU: you can use full README settings in Part 2 (num_frames=241, 736x1280).")


Thu Jul 23 07:55:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   25C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os, sys, glob, subprocess, shutil
from pathlib import Path

# --- FIX 3: never let the kernel sit in a deleted directory ---
os.chdir("/content")

REPO_ROOT = "/content/JoyAI-Echo"
OUTPUT_DIR = f"{REPO_ROOT}/inference_result"

# Echo checkpoint cache: Drive (46GB already downloaded there — reused, not re-fetched)
DRIVE_HF_CACHE = "/content/drive/MyDrive/joyai-echo-checkpoints/hf_cache"
# Gemma cache: LOCAL disk (FIX 5 — Drive FUSE kept corrupting/stalling gemma shard 5)
LOCAL_HF_CACHE = "/content/hf_cache_local"

USE_DRIVE = os.path.isdir("/content/drive/MyDrive")
if USE_DRIVE:
    os.makedirs(DRIVE_HF_CACHE, exist_ok=True)
os.makedirs(LOCAL_HF_CACHE, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

def run(cmd, cwd=None, label=""):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"--- FAILED: {label or ' '.join(cmd)} (exit {r.returncode}) ---")
        print(r.stdout[-2000:])
        print(r.stderr[-3000:])
        raise RuntimeError(f"Step failed: {label or cmd}")
    return r

# --- HF auth from Colab secret 'hf' ---
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("hf")
print("HF token loaded from Colab secret 'hf'.")

# --- FIX 4: clone only counts if requirements.txt is actually there ---
if not os.path.exists(f"{REPO_ROOT}/requirements.txt"):
    shutil.rmtree(REPO_ROOT, ignore_errors=True)
    print("Cloning repo...")
    run(["git", "clone", "https://github.com/vermasachin6102/JoyAI-Echo.git", REPO_ROOT], label="git clone")
else:
    print("Repo present and complete, skipping clone.")

for sub in ["ltx-core/src", "ltx-pipelines/src", "ltx-distillation/src"]:
    p = os.path.join(REPO_ROOT, sub)
    if p not in sys.path:
        sys.path.insert(0, p)

# --- System deps ---
print("Installing ffmpeg...")
run(["apt-get", "-qq", "update"], label="apt update")
run(["apt-get", "-qq", "install", "-y", "ffmpeg"], label="apt install ffmpeg")

# --- Python deps (pinned CUDA 12.8 stack) ---
print("Installing pinned torch stack (this takes a few minutes)...")
run(["pip", "install", "--quiet", "--index-url", "https://download.pytorch.org/whl/cu128",
     "torch==2.8.0", "torchvision==0.23.0", "torchaudio==2.8.0"], label="torch install")
print("Installing requirements.txt...")
run(["pip", "install", "--quiet", "-r", "requirements.txt"], cwd=REPO_ROOT, label="requirements.txt")
# --- FIX 2: constrained hub install AFTER requirements, never unpinned -U ---
print("Installing huggingface_hub (constrained <1.0 for transformers 4.57.6)...")
run(["pip", "install", "--quiet", "huggingface_hub[cli]>=0.34.0,<1.0"], label="hf hub install")

import torch
print("torch:", torch.__version__, "| CUDA:", torch.version.cuda, "| available:", torch.cuda.is_available())

# --- Checkpoints ---
from huggingface_hub import snapshot_download
os.makedirs(f"{REPO_ROOT}/checkpoints", exist_ok=True)

# Echo checkpoint (~46GB) — Drive cache first (instant skip if already downloaded)
print("Fetching JoyAI-Echo release checkpoint (skips instantly if Drive-cached)...")
echo_dir = snapshot_download(
    repo_id="jdopensource/JoyAI-Echo",
    cache_dir=DRIVE_HF_CACHE if USE_DRIVE else LOCAL_HF_CACHE,
    allow_patterns=["*.safetensors", "*.json", "*.md"],
)
candidates = glob.glob(os.path.join(echo_dir, "**", "*.safetensors"), recursive=True)
assert candidates, "No .safetensors found in jdopensource/JoyAI-Echo"
dst_echo = f"{REPO_ROOT}/checkpoints/echo-longvideo-release.safetensors"
if os.path.islink(dst_echo) or os.path.exists(dst_echo):
    os.remove(dst_echo)
os.symlink(candidates[0], dst_echo)
print("Echo checkpoint ->", os.path.realpath(dst_echo))

# --- FIX 1: the repo's inference.yaml actually looks for checkpoints/test.safetensors ---
dst_test = f"{REPO_ROOT}/checkpoints/test.safetensors"
if os.path.islink(dst_test) or os.path.exists(dst_test):
    os.remove(dst_test)
os.symlink(dst_echo, dst_test)
print("Config-compat symlink: test.safetensors ->", os.path.realpath(dst_test))

# Gemma (~24GB) — LOCAL disk (FIX 5). Resumable if interrupted.
print("Fetching gemma-3-12b-it to local disk (gated — needs accepted license)...")
gemma_dir = snapshot_download(repo_id="google/gemma-3-12b-it", cache_dir=LOCAL_HF_CACHE)
dst_gemma = f"{REPO_ROOT}/checkpoints/gemma-3-12b"
if os.path.islink(dst_gemma):
    os.remove(dst_gemma)
elif os.path.isdir(dst_gemma):
    shutil.rmtree(dst_gemma)
os.symlink(gemma_dir, dst_gemma)
print("Gemma ->", os.path.realpath(dst_gemma))

# Verify every file the pipeline will open actually resolves
print("\n--- Verification ---")
ok = True
for label, path in [
    ("config's checkpoint (test.safetensors)", dst_test),
    ("gemma shard 5 (the one that kept failing)", f"{dst_gemma}/model-00005-of-00005.safetensors"),
    ("gemma tokenizer", f"{dst_gemma}/tokenizer.model"),
]:
    exists = os.path.exists(path)  # follows symlinks — catches broken links
    print(("OK  " if exists else "MISSING  ") + label)
    ok = ok and exists
print("\nSETUP COMPLETE — go to Part 2." if ok else "\nSetup incomplete — re-run this cell (downloads resume).")

In [ ]:
import json, glob, subprocess, time
from pathlib import Path
from IPython.display import Video, display

def generate_video(
    prompts,
    name="my_story",
    seed=42,
    num_frames=121,
    video_height=480,
    video_width=832,
    preview=True,
    seed_video=None,   # path to a seed .mp4 (e.g. a Veo 3 clip copied to Drive) to prime the memory bank
):
    # Write prompts to JSON, run inference.py, return path to the output .mp4.
    # Full stderr is printed on failure — no more hidden tracebacks.
    prompts_dir = Path(REPO_ROOT) / "prompts"
    prompts_dir.mkdir(parents=True, exist_ok=True)
    prompt_file = prompts_dir / f"{name}.json"
    with open(prompt_file, "w") as f:
        json.dump({"prompts": prompts}, f, indent=2)
    print(f"Wrote {len(prompts)} shot(s) to {prompt_file}")

    cmd = ["python", "inference.py",
           "--prompts-glob", prompt_file.name,
           "--num-frames", str(num_frames),
           "--video-height", str(video_height),
           "--video-width", str(video_width),
           "--seed", str(seed)]
    if seed_video:
        cmd += ["--seed-video", str(seed_video)]
    print("Running:", " ".join(cmd))
    t0 = time.time()
    r = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
    print(f"inference.py exited {r.returncode} after {time.time() - t0:.0f}s")
    print("----- log tail -----")
    print("\n".join((r.stdout or "").splitlines()[-15:]))
    if r.returncode != 0:
        print("----- STDERR (last 60 lines) -----")
        print("\n".join((r.stderr or "").splitlines()[-60:]))
        return None

    outputs = sorted(glob.glob(f"{OUTPUT_DIR}/outputs/**/*.mp4", recursive=True),
                     key=lambda p: Path(p).stat().st_mtime)
    if not outputs:
        print("No .mp4 produced — check the log above.")
        return None
    latest = outputs[-1]
    print("Output video:", latest)
    if preview:
        display(Video(latest, embed=True, width=640))
    return latest

print("generate_video() ready — go to Part 2.")

In [ ]:
prompts = [
    # Shot 1 — your original scene, restructured into Echo's format
    "PIERRE is a small round sky-blue cartoon parrot with big friendly eyes, wearing a red beret and a red-and-white striped scarf, using expressive wings like hands. DINO is a chubby mint-green baby dinosaur with a stubby tail, wearing a yellow t-shirt with a sun print. Pierre has a bright, cheerful high-pitched cartoon voice with playful bouncy pacing; Dino has a warm, giggly, childlike voice. At normal speed, Pierre flies in a happy looping arc through the sky and lands gently on top of Dino's head, wings fluttering for balance; Dino giggles with delight as Pierre lands. In a bright cheerful voice, Pierre says, \"Bonjour, Dino! Today we learn French!\" The style is bright 2D cartoon animation in a Cocomelon/ChuChu TV look, thick soft outlines, rounded shapes, saturated warm colors, and gentle squash-and-stretch animation. A simple stable wide shot with a gentle push-in frames both characters, no fast cuts or shake. The background is a sunny meadow with rolling green hills, a smiling sun, puffy clouds, big daisies, and a colorful toy train with seven coaches (yellow, red, green, orange, blue, purple, pink) parked across the meadow. The toy train whistles softly and jingles gently in the background, with a cheerful, bouncy children's music bed underneath.",

    # Shot 2 — Pierre teaches the first word
    "PIERRE is the same small round sky-blue cartoon parrot with big friendly eyes, red beret, and red-and-white striped scarf, with a bright cheerful high-pitched voice. DINO is the same chubby mint-green baby dinosaur in a yellow sun-print t-shirt, with a warm giggly voice. At normal speed, Pierre hops onto Dino's snout and flaps his wings excitedly while Dino watches with wide, attentive eyes. In a bright, encouraging voice, Pierre says, \"Say it with me: Bonjour means hello!\" Dino tilts his head and repeats slowly in a happy, clumsy voice, \"Bonjour!\" The style stays bright 2D cartoon animation, thick soft outlines, rounded shapes, warm saturated colors. A closer stable medium shot keeps both faces clearly readable for expression and mouth movement. The background remains the sunny meadow with rolling hills, puffy clouds, and the colorful toy train visible behind them. A soft cheerful chime plays when Dino says the word correctly, with the same bouncy children's music bed continuing underneath.",

    # Shot 3 — second word, a little playful stumble
    "PIERRE remains the small sky-blue cartoon parrot with red beret and scarf, cheerful high voice. DINO remains the chubby mint-green dinosaur in the yellow sun-print shirt, warm giggly voice. At normal speed, Pierre flutters in a small circle above Dino's head, wings spread wide for emphasis, while Dino claps his front feet together. In a playful, patient voice, Pierre says, \"Now try: Merci means thank you!\" Dino scrunches his face in concentration and says, \"Mer... Mercii!\" then breaks into a proud giggle. The style stays the same bright 2D cartoon look with soft rounded shapes and warm colors. A medium-wide shot gently tracks the small hop and flutter, keeping both characters framed together. The meadow, daisies, and toy train remain visible in the background. A light comedic \"boing\" sound plays on Dino's stumble, followed by a cheerful giggle-friendly musical sting, with the bouncy background music continuing softly.",

    # Shot 4 — third word, celebration
    "PIERRE is still the sky-blue cartoon parrot with red beret and scarf, cheerful voice. DINO is still the mint-green baby dinosaur in the yellow sun-print shirt, warm giggly voice. At normal speed, Pierre perches on Dino's shoulder and raises both wings in celebration as Dino wiggles his stubby tail happily. In an excited, warm voice, Pierre says, \"Great job! One more: Au revoir means goodbye!\" Dino beams and shouts, \"Au revoir!\" while hopping in place with joy. The animation stays bright and 2D cartoon-styled, thick soft outlines, warm saturated palette, gentle squash-and-stretch on the hop. A stable wide shot captures the little celebratory hop with the full sunny meadow, toy train, and daisies in view. A short triumphant musical flourish plays under the cheerful children's music bed, with faint train whistle accents.",

    # Shot 5 — recap moment
    "PIERRE remains the same sky-blue cartoon parrot, cheerful voice, red beret and scarf. DINO remains the same mint-green dinosaur, warm giggly voice, yellow sun-print shirt. At normal speed, Pierre glides down from Dino's shoulder to sit on a nearby daisy, wings folded neatly, while Dino sits down in the grass, looking thoughtful and pleased. In a warm, encouraging voice, Pierre asks, \"Can you say all three words again?\" Dino grins and recites proudly, \"Bonjour... Merci... Au revoir!\" The style remains bright 2D cartoon animation, soft rounded shapes, warm cheerful colors. A calm medium shot holds both characters at ease in the grass, keeping expressions and mouth movement clearly visible. The sunny meadow, rolling hills, and toy train continue in the soft-focus background. Gentle, warm background music plays underneath with soft ambient meadow sounds — light wind and distant birdsong.",

    # Shot 6 — closing / title card moment
    "PIERRE is still the small sky-blue cartoon parrot with red beret and scarf, cheerful voice. DINO is still the chubby mint-green dinosaur in the yellow sun-print shirt, warm giggly voice. At normal speed, Pierre flies up above Dino and does one happy celebratory loop as Dino claps and laughs below, the toy train chugging gently into view behind them. In a bright, cheerful sign-off voice, Pierre says, \"Bonne journée — that means have a great day!\" Dino waves enthusiastically and giggles, \"Bonne journée!\" The style remains bright 2D cartoon animation, Cocomelon/ChuChu TV look, thick soft outlines, warm saturated colors. A simple stable wide shot frames the full happy scene with a gentle final push-in as if closing the lesson. The sunny meadow, smiling sun, puffy clouds, big daisies, and the full colorful toy train remain visible in the background. The toy train whistles cheerfully one last time as the upbeat children's music bed swells warmly to a gentle close.",
]

# TODO: set this to your Veo 3 clip's path on Drive, e.g.:
#   "/content/drive/MyDrive/veo3_clips/intro.mp4"
# Leave as None to skip seeding and generate from prompts alone.
SEED_VIDEO_PATH = None  # <-- placeholder: paste your Veo 3 Drive video path here

video_path = generate_video(
    prompts,
    name="french_lesson",
    seed=42,
    num_frames=241,      # full README setting — you're on G4 (96GB), no need to reduce
    video_height=736,
    video_width=1280,
    seed_video=SEED_VIDEO_PATH,
)

In [ ]:
import shutil, os
from google.colab import files

if video_path:
    if os.path.isdir("/content/drive/MyDrive"):
        keep_dir = "/content/drive/MyDrive/joyai-echo-outputs"
        os.makedirs(keep_dir, exist_ok=True)
        kept = shutil.copy(video_path, keep_dir)
        print("Copied to Drive:", kept)
    files.download(video_path)